#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

In [0]:
df.display()

#Silver Transformations

## Trim 

In [0]:
#Instead of reconstructing the DataFrame plan for each column using a for loop, this method updates all string columns in a single plan, making it faster.
#.alias(file.name) - keeps the origianl name of the column instead of "trim(prd_id)" for example
df = df.select([
    trim(col(field.name)).alias(field.name) if isinstance(field.dataType, StringType) 
    else col(field.name) 
    for field in df.schema.fields
])


#Key & ID Parsing: Extract & Clean the Category ID & Product Key

In [0]:
#extracts the first five characters from the prd_key column, replaces any dashes with underscores, and saves the cleaned result into a new column named cat_id.
df = df.withColumn ("cat_id", F.regexp_replace(F.substring(col("prd_key"),1,5), "-", "_"))
# Strips the first six characters (like "M-101-") from prd_key to isolate the remaining core item identifier as the "prd_key" (like "BIKE-999").
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

#This is important so we could later join tables with the "cat_id" and "prd_key" columns.

##Cost Cleanup

In [0]:
#If there are any null values in the prd_cost column, replace them with zero. 
# Keep in mind to run this code before you rename the columns

df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

#Product Line Normalization 

In [0]:
#Normalize product line to clarify it and make it consistent since there are various names for the same product line 
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise ("n/a")
    )
)

#Date Casting

In [0]:
#Since prd_start_dt & prd_end_dt is set to a timestamp datatype and it's only showing zero's we might as well just keep it as a date type
df = df.withColumn("prd_start_dt", col("prd_end_dt").cast("date"))

#Tracking full Type 2 SCD history by using LEAD to calculate prd_end_dt.

In [0]:
# 1. Define the window partition and order
window_spec = Window.partitionBy("prd_key").orderBy("prd_start_dt")

# 2. Calculate the end date using LEAD and date_sub
#In PySpark, performing date_sub on a Date column automatically preserves the DateType; no need for casting
df = df.withColumn(
    "prd_end_dt",
    F.date_sub(F.lead("prd_start_dt", 1).over(window_spec), 1)
)

#Renaming Columns

In [0]:
#We're also renaming the created column "cat_id" to category_id from our parsing code so other tables with "cat_id" can be joined with this table 
#Using a dictionary to map the old column names to the new ones
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


#Sanity checks of dataframe

In [0]:
df.limit(20).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

#Sanity checks of the Silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10

In [0]:
%sql
-- Drop table if needed for changes or 
DROP TABLE IF EXISTS workspace.silver.crm_products;